In [ ]:
import os
from pathlib import Path

def find_project_root() -> Path:
    explicit_root = os.environ.get("VOCIFERA_ROOT")
    if explicit_root:
        candidate = Path(explicit_root).expanduser().resolve()
        if (candidate / "F5-TTS" / "pyproject.toml").exists():
            return candidate

    search_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    search_roots.extend([
        Path("/kaggle/working/vocifera"),
        Path("/content/vocifera"),
    ])

    for candidate in search_roots:
        if (candidate / "F5-TTS" / "pyproject.toml").exists():
            return candidate

    raise FileNotFoundError(
        "No pude localizar la carpeta del proyecto. Define VOCIFERA_ROOT o ejecuta el notebook dentro del repo."
    )

PROJECT_ROOT = find_project_root()
F5_ROOT = PROJECT_ROOT / "F5-TTS"
OUTPUT_ROOT = Path(
    os.environ.get(
        "VOCIFERA_OUTPUT_DIR",
        "/kaggle/working" if Path("/kaggle/working").exists() else str(PROJECT_ROOT),
    )
).resolve()

HF_REPO_ID = os.environ.get("OPENF5_HF_REPO_ID", "mrfakename/Open-F5-TTS")
HF_FILENAME = os.environ.get("OPENF5_HF_FILENAME", "model_last.pt")
DATASET_NAME = os.environ.get("OPENF5_DATASET_NAME", "OpenF5_ES")
METADATA_CSV = Path(
    os.environ.get("OPENF5_METADATA_CSV", "/kaggle/input/tu-dataset-subido/metadata.csv")
).expanduser().resolve()

BASE_CKPT_DIR = PROJECT_ROOT / "ckpts" / "Open-F5"
BASE_CKPT_PATH = BASE_CKPT_DIR / HF_FILENAME
PREPARED_DATASET_DIR = F5_ROOT / "data" / f"{DATASET_NAME}_pinyin"
CHECKPOINT_DIR = F5_ROOT / "ckpts" / DATASET_NAME

print(f"Project root: {PROJECT_ROOT}")
print(f"F5-TTS root: {F5_ROOT}")
print(f"Output root: {OUTPUT_ROOT}")
print(f"HF repo: {HF_REPO_ID}")
print(f"Base checkpoint target: {BASE_CKPT_PATH}")
print(f"Metadata CSV: {METADATA_CSV}")
print(f"Prepared dataset dir: {PREPARED_DATASET_DIR}")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")

In [ ]:
from huggingface_hub import hf_hub_download

BASE_CKPT_DIR.mkdir(parents=True, exist_ok=True)
downloaded_checkpoint = Path(
    hf_hub_download(
        repo_id=HF_REPO_ID,
        filename=HF_FILENAME,
        local_dir=BASE_CKPT_DIR,
    )
).resolve()

print(f"Checkpoint descargado en: {downloaded_checkpoint}")

In [ ]:
import os
import subprocess
import sys

if not METADATA_CSV.exists():
    raise FileNotFoundError(f"No existe el metadata.csv: {METADATA_CSV}")

PREPARED_DATASET_DIR.parent.mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
src_path = str((F5_ROOT / "src").resolve())
env["PYTHONPATH"] = src_path if not env.get("PYTHONPATH") else src_path + os.pathsep + env["PYTHONPATH"]

prepare_cmd = [
    sys.executable,
    str(F5_ROOT / "src" / "f5_tts" / "train" / "datasets" / "prepare_csv_wavs.py"),
    str(METADATA_CSV),
    str(PREPARED_DATASET_DIR),
]

print("Ejecutando:", " ".join(prepare_cmd))
subprocess.run(prepare_cmd, cwd=F5_ROOT, env=env, check=True)

In [ ]:
import os
import shutil
import subprocess
import sys

if not BASE_CKPT_PATH.exists():
    raise FileNotFoundError(f"No existe el checkpoint base: {BASE_CKPT_PATH}")

env = os.environ.copy()
src_path = str((F5_ROOT / "src").resolve())
env["PYTHONPATH"] = src_path if not env.get("PYTHONPATH") else src_path + os.pathsep + env["PYTHONPATH"]

train_cmd = [sys.executable]
if shutil.which("accelerate"):
    train_cmd.extend(["-m", "accelerate.commands.launch"] )

train_cmd.extend([
    str(F5_ROOT / "src" / "f5_tts" / "train" / "finetune_cli.py"),
    "--exp_name", "F5TTS_v1_Base",
    "--dataset_name", DATASET_NAME,
    "--finetune",
    "--pretrain", str(BASE_CKPT_PATH),
    "--learning_rate", "1e-5",
    "--batch_size_type", "sample",
    "--batch_size_per_gpu", "2",
    "--grad_accumulation_steps", "4",
    "--epochs", "100",
    "--save_per_updates", "2000",
    "--last_per_updates", "1000",
    "--keep_last_n_checkpoints", "3",
])

print("Ejecutando:", " ".join(train_cmd))
subprocess.run(train_cmd, cwd=F5_ROOT, env=env, check=True)

In [ ]:
import shutil

destino_zip = OUTPUT_ROOT / f"{DATASET_NAME.lower()}_checkpoints"

if CHECKPOINT_DIR.exists():
    shutil.make_archive(destino_zip.as_posix(), "zip", CHECKPOINT_DIR.as_posix())
    print(f"Checkpoints comprimidos en: {destino_zip}.zip")
else:
    print(f"No existe la carpeta de checkpoints: {CHECKPOINT_DIR}")